# 2. Procesamiento ETL: BANCI (Secretariado Ejecutivo)

**Objetivo:** Descargar, limpiar, estandarizar y aplicar filtros a la base de datos de BANCI para su posterior cruce con el RNPDNO.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
import importlib

# Configuración de rutas
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import config

from src import cleaning, validacion_banci
from src.utils import standardize_columns
importlib.reload(config)
importlib.reload(cleaning)
importlib.reload(validacion_banci)

print(f"DEBUG: Config reload complete.")

if hasattr(config, 'ESTADO') and config.ESTADO:
    nombre_estado = config.ENTIDADES.get(config.ESTADO, "Desconocido")
    print(f"ℹ️ MODO FILTRO POR ESTADO ACTIVO: {nombre_estado} (ID: {config.ESTADO})")
else:
    print("ℹ️ MODO GLOBAL: Procesando todo el país.")

In [ ]:
# 1. Funciones de limpieza y filtrado de columnas
def clean_col_text(col_name):
    """Normaliza un nombre de columna individual."""
    return str(col_name).lower().strip().replace(" ", "_").replace("(", "").replace(")", "")

def clean_col_names(df):
    """Limpia los nombres de las columnas de un DataFrame."""
    df.columns = [clean_col_text(c) for c in df.columns]
    return df

# Preparar columnas esperadas
COLS_BANCI_TARGET = set(config.BANCI_COLUMN_MAPPING.keys())
valid_banci_inputs = set()
for target in COLS_BANCI_TARGET: 
    valid_banci_inputs.add(target)
    for map_key, map_variations in config.BANCI_COLUMN_MAPPING.items():
        if target in map_variations:
            variations = [clean_col_text(v) for v in map_variations]
            valid_banci_inputs.update(variations)

def filter_banci_cols(col_name):
    """Filtra columnas de BANCI antes de cargar el Excel completo."""
    if not isinstance(col_name, str): return False
    norm = clean_col_text(col_name)
    return norm in valid_banci_inputs

banci_sheet_desc = f", hoja: '{config.BANCI_SHEET}'" if (hasattr(config, 'BANCI_SHEET') and config.BANCI_SHEET is not None) else " (hoja predeterminada)"
print(f"Intentando cargar BANCI desde: {config.BANCI_FILE}{banci_sheet_desc}")

In [ ]:
# Función extraída de limpiar_banci.ipynb para corregir fechas en el futuro
def corregir_anio_futuro(df, columna_fecha, fecha_limite):
    """
    Corrige los años de fechas futuras restando años hasta que la fecha
    sea menor o igual a fecha_limite.
    """
    if columna_fecha not in df.columns:
        return df
        
    df[columna_fecha] = pd.to_datetime(df[columna_fecha], errors='coerce')
    fecha_limite = pd.to_datetime(fecha_limite)

    def corregir_fecha(fecha):
        if pd.isnull(fecha):
            return fecha
        while fecha > fecha_limite:
            try:
                fecha = fecha.replace(year=fecha.year - 1)
            except ValueError:
                # Para fechas como 29 de febrero en años no bisiestos
                fecha = fecha.replace(month=2, day=28, year=fecha.year - 1)
        return fecha

    df[columna_fecha] = df[columna_fecha].apply(corregir_fecha)
    return df


In [ ]:
from src.matching import normalizar_texto, reparar_nombres_mal_formados
from src.cleaning import es_curp_valida_vectorizado, limpiar_curp
import re

# 2. Cargar y Limpiar BANCI
try:
    print("  ⚡ Aplicando filtro de columnas 'usecols' en BANCI...")
    try:
        banci_sheet_name = config.BANCI_SHEET if (hasattr(config, 'BANCI_SHEET') and config.BANCI_SHEET is not None) else 0
        df_banci = pd.read_excel(config.BANCI_FILE, sheet_name=banci_sheet_name, usecols=filter_banci_cols)
        print(f"  ✅ Carga optimizada exitosa: {len(df_banci):,} registros.")
    except Exception as e:
        print(f"  ⚠️ Falló usecols en BANCI: {e}. Cargando todo el archivo (lento)...")
        banci_sheet_name = config.BANCI_SHEET if (hasattr(config, 'BANCI_SHEET') and config.BANCI_SHEET is not None) else 0
        df_banci = pd.read_excel(config.BANCI_FILE, sheet_name=banci_sheet_name)
    
    # Normalización inicial de nombres de columnas
    df_banci = clean_col_names(df_banci)
    
    # Estandarizar columnas a los nombres objetivo en config. Va antes de la REGLA B
    # para que la copia `_banci` se llame como el nombre estándar y no como el
    # encabezado del Excel: junio de 2026 trae 'ENTIDAD FEDERATIVA' y agosto 'Entidad',
    # y BANCI_OUTPUT_COLUMNS pide `entidad_banci` en los dos. Sólo renombra: el valor
    # de la copia sigue siendo el que vino.
    df_banci = standardize_columns(df_banci, config.BANCI_COLUMN_MAPPING, strict=False, verbose=True)

    # --- REGLA B: Preservar columnas originales intactas con sufijo _banci ---
    print("  📌 Preservando copias de columnas originales con sufijo _banci...")
    for col in list(df_banci.columns):
        if not col.endswith('_banci'):
            df_banci[f"{col}_banci"] = df_banci[col].copy()

    # Validar que existan las columnas de BANCI (las columnas de delitos son opcionales)
    # `nom_mun_hchos` e `id_ci` son opcionales: los usa el motor refinado, pero no
    # todos los cortes de BANCI los traen y su ausencia no invalida el ETL.
    _COLS_BANCI_OPCIONALES = ['fue_delito', 'delito_hom', 'nom_mun_hchos', 'id_ci']
    cols_requeridas = [col for col in COLS_BANCI_TARGET if col not in _COLS_BANCI_OPCIONALES]
    cleaning.validar_columnas(df_banci, cols_requeridas, "BANCI")

    # Contenido de la entrega y comparación con la última procesada: más registros que
    # la anterior, corte más reciente, ninguna entidad que desaparezca… (ver el módulo).
    # Va antes de derivar nada y sobre el archivo completo, sin el filtro de ESTADO.
    resumen_banci = validacion_banci.validar(
        df_banci, config.BANCI_FILE,
        referencia=validacion_banci.leer_referencia(config.ETL_BANCI_OUTPUT_FILE),
        estricta=config.VALIDACION_BANCI_ESTRICTA,
    )

    # --- REGLA C: Generación de Columnas Derivadas y Limpias (_derivada_banci) ---
    print("\n--- Generando variables limpias y derivadas (_derivada_banci) ---")
    
    # 1. Reparación de nombres mal formados y división de nombres/apellidos
    df_banci = reparar_nombres_mal_formados(df_banci)
    
    for c in ['nombres', 'primer_apellido', 'segundo_apellido']:
        if c in df_banci.columns:
            df_banci[c] = df_banci[c].astype(str).replace(r'^0$', np.nan, regex=True).replace('nan', np.nan)
            df_banci[f"{c}_derivada_banci"] = df_banci[c].copy()

    # Nombre completo normalizado y es_nombre_valido
    col_b_nombres = 'nombres'
    col_b_primer_apellido = 'primer_apellido'
    col_b_segundo_apellido = 'segundo_apellido'
    
    nom_raw = (
        df_banci[col_b_nombres].fillna('').astype(str) + " " + 
        df_banci[col_b_primer_apellido].fillna('').astype(str) + " " + 
        df_banci[col_b_segundo_apellido].fillna('').astype(str)
    )
    df_banci['nombre_completo_derivada_banci'] = normalizar_texto(nom_raw)
    
    patron_ruido = r"SE DESCONOCE|RESTOS OSEOS|CUERPO|SIN VIDA|DESCONOCIDO|NO IDENTIFICADO|\bSD\b|\bNA\b|^M$|([a-z]\.){2,}[a-z]?"
    s_nom = df_banci['nombre_completo_derivada_banci'].fillna('')
    cond_not_na = df_banci['nombre_completo_derivada_banci'].notna()
    cond_no_digit = ~s_nom.str.contains(r'\d', regex=True)
    cond_no_ruido = ~s_nom.str.contains(patron_ruido, flags=re.IGNORECASE, regex=True)
    cond_len = s_nom.str.len() >= 11
    df_banci['es_nombre_valido_derivada_banci'] = cond_not_na & cond_no_digit & cond_no_ruido & cond_len

    # 2. Validez de la CURP. La original queda intacta en `curp_banci` (REGLA B). No se
    # guarda una CURP "limpia" aparte: sería idéntica, y el motor estándar la limpia al
    # comparar con esta misma función.
    if 'curp' in df_banci.columns:
        df_banci['curp_valida_derivada_banci'] = es_curp_valida_vectorizado(
            limpiar_curp(df_banci['curp']).astype(str))
    else:
        df_banci['curp_valida_derivada_banci'] = False

    # 3. Fechas derivadas (Datetime y corrección de fechas futuras)
    fecha_corte = datetime.now().strftime('%Y-%m-%d')
    
    if 'fha_de_hchos' in df_banci.columns:
        df_banci['fha_de_hchos_derivada_banci'] = pd.to_datetime(df_banci['fha_de_hchos'], errors='coerce')
        df_banci = corregir_anio_futuro(df_banci, 'fha_de_hchos_derivada_banci', fecha_corte)
    else:
        df_banci['fha_de_hchos_derivada_banci'] = pd.NaT

    if 'fha_nac' in df_banci.columns:
        df_banci['fha_nac_derivada_banci'] = pd.to_datetime(df_banci['fha_nac'], errors='coerce')
    else:
        df_banci['fha_nac_derivada_banci'] = pd.NaT

    # 4. Llaves e Identificadores (Casteo a string)
    for col_std, der_col in [('id_mun_hchos', 'id_mun_hchos_derivada_banci'), 
                             ('llave', 'llave_derivada_banci'), 
                             ('folio_rnpdno', 'fub_derivada_banci')]:
        if col_std in df_banci.columns:
            df_banci[der_col] = df_banci[col_std].astype(str).replace('nan', np.nan).replace('None', np.nan)
        elif f"{col_std}_banci" in df_banci.columns:
            df_banci[der_col] = df_banci[f"{col_std}_banci"].astype(str).replace('nan', np.nan).replace('None', np.nan)
        else:
            df_banci[der_col] = np.nan
        df_banci[col_std] = df_banci[der_col]

    # 5. Entidad, Estatus y Delitos
    col_estatus = next((k for k, v in config.BANCI_COLUMN_MAPPING.items() if 'estatus' in [x.lower() for x in v] or k == 'localizado_o_no_localizado'), 'localizado_o_no_localizado')
    if col_estatus in df_banci.columns:
        df_banci['localizado_o_no_localizado_derivada_banci'] = df_banci[col_estatus].astype(str).str.strip().str.upper()
    else:
        df_banci['localizado_o_no_localizado_derivada_banci'] = 'NO LOCALIZADA'

    if 'entidad' in df_banci.columns:
        df_banci['entidad_derivada_banci'] = normalizar_texto(df_banci['entidad'])
    else:
        df_banci['entidad_derivada_banci'] = None

    # 6. Número de Carpeta de Investigación → ntra_ci_derivada_banci
    # NTRA_CI es el número real de carpeta (ej. 'JALISCO/FGE/2024/001234').
    # Sin esta columna el motor refinado solo matchea por CURP/nombre, sin folio,
    # produciendo muchos menos matches que normalizar_folios_v6.py.
    for _col_ntra in ['NTRA_CI', 'ntra_ci', 'NtraCi', 'ntraci', 'ntra_ci_banci']:
        if _col_ntra in df_banci.columns:
            df_banci['ntra_ci_derivada_banci'] = (
                df_banci[_col_ntra].astype(str).str.strip()
                .replace({'nan': np.nan, 'None': np.nan, '<NA>': np.nan, '': np.nan})
            )
            n_nulos = df_banci['ntra_ci_derivada_banci'].isna().sum()
            print(f'  ntra_ci_derivada_banci desde {_col_ntra!r}: {len(df_banci)-n_nulos:,} no nulos')
            break
    else:
        df_banci['ntra_ci_derivada_banci'] = np.nan
        print('  NTRA_CI no encontrada: motor refinado no hará folio-match.')

    # 6b. ID_CI → id_ci_derivada_banci
    # normalizar_folios_v6.py usa este identificador como consecutivo de respaldo
    # cuando el de NTRA_CI tiene <= 2 dígitos: los consecutivos cortos (o vacíos)
    # no son comparables y sin este rescate ~1 de cada 3 registros no localizados
    # nunca puede alcanzar "folio limpio identico".
    for _col_idci in ['ID_CI', 'id_ci', 'idci', 'id_ci_banci']:
        if _col_idci in df_banci.columns:
            df_banci['id_ci_derivada_banci'] = (
                df_banci[_col_idci].astype(str).str.strip()
                .replace({'nan': np.nan, 'None': np.nan, '<NA>': np.nan, '': np.nan})
            )
            n_nulos = df_banci['id_ci_derivada_banci'].isna().sum()
            print(f'  id_ci_derivada_banci desde {_col_idci!r}: {len(df_banci)-n_nulos:,} no nulos')
            break
    else:
        df_banci['id_ci_derivada_banci'] = np.nan
        print('  ID_CI no encontrada: los folios con consecutivo corto no tendrán respaldo.')

    # 6c. NOM_MUN_HCHOS → nom_mun_hchos_derivada_banci
    # El motor refinado compara municipios con fuzzy match (UMBRAL_MUNICIPIO = 0.80),
    # una métrica para NOMBRES. Las claves numéricas de las dos bases no comparten
    # codificación (su intersección literal es cero), así que sin el nombre el
    # municipio nunca coincide y se pierde como corroborador.
    for _col_mun in ['nom_mun_hchos', 'NOM_MUN_HCHOS', 'municipio_hechos', 'municipio']:
        if _col_mun in df_banci.columns:
            df_banci['nom_mun_hchos_derivada_banci'] = normalizar_texto(df_banci[_col_mun])
            n_nulos = df_banci['nom_mun_hchos_derivada_banci'].isna().sum()
            print(f'  nom_mun_hchos_derivada_banci desde {_col_mun!r}: {len(df_banci)-n_nulos:,} no nulos')
            break
    else:
        df_banci['nom_mun_hchos_derivada_banci'] = np.nan
        print('  NOM_MUN_HCHOS no encontrada: el municipio no se usará como corroborador.')

    # La construcción replica la del SESNSP: ENTIDAD + '/' + carpeta, usando ID_CI
    # cuando la carpeta no es un folio real (vacía o puros ceros). Verificado contra
    # `Base_BANCI_con_match_..._24082026.xlsx`: reproduce la MISMA partición que su
    # `id_carpeta` (4,715 grupos idénticos, 0 registros con colectividad distinta).
    # El string no tiene que ser idéntico al de ellos: sólo se usa para agrupar.
    _carp = df_banci['ntra_ci_derivada_banci'].fillna('').astype(str).str.strip()
    _idci = df_banci['id_ci_derivada_banci'].fillna('').astype(str).str.strip()
    _carp_real = ~(_carp.eq('') | _carp.str.fullmatch(r'0+', na=False))
    _parte = normalizar_texto(_carp.where(_carp_real, _idci))
    _ent = normalizar_texto(df_banci['entidad_derivada_banci'])
    df_banci['id_carpeta_derivada_banci'] = (_ent + '/' + _parte).where(_parte.str.strip() != '', np.nan)
    _n = int(df_banci['id_carpeta_derivada_banci'].notna().sum())
    print(f'  id_carpeta_derivada_banci: {_n:,} construidos '
          f'({int((~_carp_real).sum()):,} desde ID_CI por carpeta vacía o en ceros)')

    # 7. Flag de Skip Match (Identidad Inválida)
    col_fub_check = 'fub_derivada_banci'
    cond_identidad_invalida = (~df_banci['curp_valida_derivada_banci']) & (~df_banci['es_nombre_valido_derivada_banci'])
    
    if col_fub_check in df_banci.columns:
        cond_salvados_por_fub = df_banci[col_fub_check].notna() & (df_banci[col_fub_check] != '') & (df_banci[col_fub_check] != 'nan')
        mask_invalido = cond_identidad_invalida & (~cond_salvados_por_fub)
    else:
        mask_invalido = cond_identidad_invalida

    df_banci['skip_match_derivada_banci'] = False
    df_banci.loc[mask_invalido, 'skip_match_derivada_banci'] = True
    df_banci['comentario_match_derivada_banci'] = None
    df_banci.loc[mask_invalido, 'comentario_match_derivada_banci'] = 'Identidad Inválida (Sin Nombre ni CURP válidos)'

    # Filtro MODO ESTADO (si está configurado)
    if hasattr(config, 'ESTADO') and config.ESTADO:
        estado_filtro = config.ENTIDADES.get(config.ESTADO, "").upper()
        if 'entidad_derivada_banci' in df_banci.columns:
            print(f"  🔍 Filtrando BANCI por '{estado_filtro}' en 'entidad_derivada_banci'...")
            ent_vals = df_banci['entidad_derivada_banci'].fillna('')
            estado_filtro_norm = normalizar_texto(estado_filtro)
            mask = ent_vals == estado_filtro_norm
            n_antes = len(df_banci)
            df_banci = df_banci[mask].copy()
            n_despues = len(df_banci)
            print(f"    🔻 BANCI Filtrado: {n_antes} -> {n_despues} registros.")

    # --- FORZAR HOMOGENEIZACIÓN DE TIPOS EN COLUMNAS OBJECT PARA PYARROW ---
    print("\n🔨 Homogeneizando tipos en columnas object para exportación a Parquet...")
    for col_name in df_banci.columns:
        if df_banci[col_name].dtype == 'object':
            df_banci[col_name] = df_banci[col_name].astype(str).replace(r'^(nan|None|<NA>|NAN|NONE)$', np.nan, regex=True)
            df_banci[col_name] = df_banci[col_name].where(df_banci[col_name].notna() & (df_banci[col_name] != 'nan'), np.nan)

    # Sólo sale el contrato de config.BANCI_OUTPUT_COLUMNS: las copias `_banci` tal
    # como vinieron y las `_derivada_banci`. Las columnas de trabajo sin sufijo se
    # quedan en este notebook.
    df_banci = df_banci[config.BANCI_OUTPUT_COLUMNS]

    # Exportar resultados Parquet
    if not df_banci.empty:
        out_banci = config.ETL_BANCI_OUTPUT_FILE
        # Con el resumen de la entrega en los metadatos: es la referencia contra la que
        # se compara la siguiente (src/validacion_banci.py).
        validacion_banci.escribir_parquet(df_banci, out_banci, resumen_banci)
        print(f"\n✅ Archivo BANCI limpio guardado en: {out_banci}")

except Exception as e:
    raise RuntimeError(f"❌ ERROR CRÍTICO cargando/procesando BANCI: {e}")